# Fashionista: Colab Master Notebook

Efficient GPU pipeline for all versions (V0a–V4). Each section is self-contained — run them in order or skip to what you need.

**GPU strategy**: Load one model at a time → process all images → save to disk → unload → load next. This keeps GPU at ~100% utilization with zero idle time.

**Colab Pro**: T4/V100/A100 GPU. ~24h max runtime. All intermediates checkpointed to Google Drive.

## 0. Setup & Clone

In [ ]:
# Clone repo and install dependencies
import os

if not os.path.exists('fashionista'):
    !git clone https://github.com/samudraneel05/fashionista.git

%cd fashionista
!pip install -q -r requirements.txt

# Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/fashionista'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs('data/fashionpedia', exist_ok=True)
os.makedirs('indexes', exist_ok=True)
print(f'Drive: {DRIVE_DIR}')

In [ ]:
# Download Fashionpedia (val/test split ~8K images for dev, or full ~48K)
DOWNLOAD_FULL = False  # Set True for full dataset

if not os.path.exists('data/fashionpedia/val_test'):
    !bash scripts/download_data.sh{' --full' if DOWNLOAD_FULL else ''}
else:
    print('Dataset already downloaded.')

# Count images
from utils.image_utils import get_image_paths
image_paths = get_image_paths('data/fashionpedia')
print(f'Total images: {len(image_paths)}')

## 1. V0a: Vanilla CLIP Baseline (~15 min)

Single 512-d embedding per image. Fastest to index.

In [ ]:
# Index V0a
import sys, gc
sys.path.insert(0, '.')

from indexer.clip_indexer import CLIPIndexer
from utils.image_utils import get_image_paths

image_paths = get_image_paths('data/fashionpedia')
indexer = CLIPIndexer(data_dir='data/fashionpedia', output_dir='indexes/v0a', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=64)

# Save to Drive
!cp -r indexes/v0a "{DRIVE_DIR}/"
print('V0a index saved to Drive')

# Free GPU
del indexer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Quick test: retrieve with V0a
from retriever.clip_retriever import CLIPRetriever

retriever = CLIPRetriever('v0a', 'indexes/v0a', 'data/fashionpedia', device='cuda')
results = retriever.retrieve('A person in a bright yellow raincoat', top_k=5)
for r in results:
    print(f"  {r['image_id']}: {r['score']:.4f}")

del retriever
gc.collect()
torch.cuda.empty_cache()

## 2. V0b: Marqo-FashionSigLIP (~15 min)

Domain-specific fashion CLIP. Same architecture as V0a, better embeddings.

In [ ]:
from indexer.fashion_clip_indexer import FashionCLIPIndexer

image_paths = get_image_paths('data/fashionpedia')
indexer = FashionCLIPIndexer(data_dir='data/fashionpedia', output_dir='indexes/v0b', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=64)

!cp -r indexes/v0b "{DRIVE_DIR}/"
print('V0b index saved to Drive')

del indexer
gc.collect()
torch.cuda.empty_cache()

## 3. V1: Multi-Vector Decomposition (~45 min)

3 vectors per image (global, scene, action). Uses YOLOv8 + Marqo.

In [ ]:
from indexer.multi_vector_indexer import MultiVectorIndexer

image_paths = get_image_paths('data/fashionpedia')
indexer = MultiVectorIndexer(data_dir='data/fashionpedia', output_dir='indexes/v1', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=32)

!cp -r indexes/v1 "{DRIVE_DIR}/"
print('V1 index saved to Drive')

del indexer
gc.collect()
torch.cuda.empty_cache()

## 4. V2: Region-Level Segmentation (~60 min)

Per-garment embeddings using SegFormer B2. 8 channels per image.

In [ ]:
from indexer.region_indexer import RegionIndexer

image_paths = get_image_paths('data/fashionpedia')
indexer = RegionIndexer(data_dir='data/fashionpedia', output_dir='indexes/v2', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=16)

!cp -r indexes/v2 "{DRIVE_DIR}/"
print('V2 index saved to Drive')

del indexer
gc.collect()
torch.cuda.empty_cache()

## 5. V3: VLM Captioning (~90 min)

Florence-2 + BLIP captions → BGE text embeddings. Most GPU-intensive indexing.

In [ ]:
from indexer.vlm_indexer import VLMIndexer

image_paths = get_image_paths('data/fashionpedia')
indexer = VLMIndexer(data_dir='data/fashionpedia', output_dir='indexes/v3', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=16)

!cp -r indexes/v3 "{DRIVE_DIR}/"
print('V3 index + captions saved to Drive')

del indexer
gc.collect()
torch.cuda.empty_cache()

## 6. V4: Hybrid Ensemble (~3 hours total)

Combines all: 5 CLIP vectors + 1 BGE caption vector. Uses all models sequentially.

In [ ]:
from indexer.hybrid_indexer import HybridIndexer

image_paths = get_image_paths('data/fashionpedia')
indexer = HybridIndexer(data_dir='data/fashionpedia', output_dir='indexes/v4', device='cuda')
indexer._image_paths = image_paths
indexer.index(image_paths, batch_size=16)

!cp -r indexes/v4 "{DRIVE_DIR}/"
print('V4 index + captions saved to Drive')

del indexer
gc.collect()
torch.cuda.empty_cache()

## 7. Evaluation & Comparison

Run all 100 queries across all indexed versions. Generates comparison tables, plots, and run history.

In [ ]:
# Restore indexes from Drive if needed (e.g., after session reconnect)
import shutil
for version in ['v0a', 'v0b', 'v1', 'v2', 'v3', 'v4']:
    src = f"{DRIVE_DIR}/{version}"
    dst = f"indexes/{version}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'Restored {version} from Drive')
    elif os.path.exists(dst):
        print(f'{version} already present')
    else:
        print(f'{version} not found — index it first')

In [ ]:
# Evaluate all versions
# First with 5 base queries for quick check
!python scripts/evaluate.py --all --base_only --index_dir indexes --data_dir data/fashionpedia \
    --annotations data/fashionpedia/instances_attributes_val2020.json \
    --output evaluation/results_base.json

In [ ]:
# Full evaluation with all 100 queries
!python scripts/evaluate.py --all --index_dir indexes --data_dir data/fashionpedia \
    --annotations data/fashionpedia/instances_attributes_val2020.json \
    --captions indexes/v4/captions.json \
    --output evaluation/results.json

In [ ]:
# Run ablation studies on V4
!python -m evaluation.ablation --index_dir indexes --data_dir data/fashionpedia \
    --annotations data/fashionpedia/instances_attributes_val2020.json \
    --captions indexes/v4/captions.json \
    --output evaluation/ablation_results.json

In [ ]:
# Generate all plots
!python -m evaluation.visualize --results evaluation/results.json \
    --ablations evaluation/ablation_results.json \
    --output_dir evaluation/plots

# Copy plots to Drive
!cp -r evaluation/plots "{DRIVE_DIR}/"
!cp evaluation/results.json "{DRIVE_DIR}/"
!cp evaluation/ablation_results.json "{DRIVE_DIR}/"
!cp evaluation/run_history.json "{DRIVE_DIR}/" 2>/dev/null || true
print('All results + plots saved to Drive')

In [ ]:
# Display plots inline
from IPython.display import Image, display
import os

plot_dir = 'evaluation/plots'
if os.path.exists(plot_dir):
    for fname in sorted(os.listdir(plot_dir)):
        if fname.endswith('.png'):
            print(f'\n=== {fname} ===')
            display(Image(filename=os.path.join(plot_dir, fname)))

In [ ]:
# View run history
from utils.run_tracker import RunTracker
tracker = RunTracker()
print(tracker.summary_table())

## 8. Gradio Web UI

Launch interactive demo. Use `share=True` for public link.

In [ ]:
# Launch Gradio UI with all versions
!python app/gradio_ui.py --versions v0a v0b v1 v2 v3 v4 \
    --index_dir indexes --data_dir data/fashionpedia --share

## 9. Resume After Disconnect

Colab sessions disconnect after ~12-24h. Run this cell to restore all indexes from Drive and continue.

In [ ]:
# Quick restore after reconnect
import os, shutil, sys
sys.path.insert(0, '.')

DRIVE_DIR = '/content/drive/MyDrive/fashionista'

# Re-clone if needed
if not os.path.exists('fashionista'):
    !git clone https://github.com/Samudraneel05/fashionista.git
    %cd fashionista
    !pip install -q -r requirements.txt
elif not os.path.exists('indexer'):
    %cd fashionista
    !pip install -q -r requirements.txt

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Restore indexes
for version in ['v0a', 'v0b', 'v1', 'v2', 'v3', 'v4']:
    src = f"{DRIVE_DIR}/{version}"
    dst = f"indexes/{version}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f'Restored {version}')
    elif os.path.exists(dst):
        print(f'{version} present')
    else:
        print(f'{version} missing')

# Restore results
for fname in ['results.json', 'ablation_results.json', 'run_history.json']:
    src = f"{DRIVE_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f'evaluation/{fname}')
        print(f'Restored {fname}')

print('\nReady to continue!')